# DME-Encoder Forecast Pipeline - Kaggle Notebook

This notebook trains the profile-token, multi-pooling, hybrid denoising plus forecast DME pipeline, then fine-tunes the encoder on the downstream classification target.

It is intentionally script-driven for the long training stages, while keeping setup, data preparation, smoke checks, and result aggregation visible in notebook cells.

In [ ]:
# Cell 1 - Setup & Install
import json
import pathlib
import pickle
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

WORKING_DIR = pathlib.Path("/kaggle/working")
REPO_CANDIDATES = [
    WORKING_DIR / "denoising-event-sequences",
    pathlib.Path.cwd(),
    pathlib.Path.cwd().parent,
]
REPO_DIR = next(
    (
        p
        for p in REPO_CANDIDATES
        if (p / "src").exists() and (p / "scripts").exists()
    ),
    REPO_CANDIDATES[0],
)
if not REPO_DIR.exists():
    raise FileNotFoundError(f"Repository not found. Checked: {REPO_CANDIDATES}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--quiet"],
    check=True,
)
sys.path.insert(0, str(REPO_DIR))

required_imports = {
    "yaml": "pyyaml",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "torchmetrics": "torchmetrics",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "catboost": "catboost",
}
missing_packages = []
for module_name, package_name in required_imports.items():
    try:
        __import__(module_name)
    except ImportError:
        missing_packages.append(package_name)
if missing_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", *missing_packages, "--quiet"],
        check=True,
    )

import numpy as np
import pandas as pd
import torch
import yaml

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"

print(f"Repo       : {REPO_DIR}")
print(f"Device     : {device}")
print(f"Mixed prec : {USE_AMP}")
print(f"PyTorch    : {torch.__version__}")

In [ ]:
# Cell 2 - Paths & Runtime Knobs
DATA_ROOT = WORKING_DIR / "data"
PROCESSED_DIR = DATA_ROOT / "processed" / "rosbank_forecast"
OUTPUT_DIR = WORKING_DIR / "outputs"
LOG_DIR = OUTPUT_DIR / "logs"
FORECAST_OUTPUT_DIR = OUTPUT_DIR / "forecast_pretrain"
FROZEN_OUTPUT_DIR = OUTPUT_DIR / "finetune_frozen"
FULL_OUTPUT_DIR = OUTPUT_DIR / "finetune_full"
CONFIG_PATH = WORKING_DIR / "forecast_pipeline.yaml"
RAW_PARQUET_PATH = DATA_ROOT / "rosbank_forecast_raw.parquet"

for path in [DATA_ROOT, PROCESSED_DIR, OUTPUT_DIR, LOG_DIR, FORECAST_OUTPUT_DIR, FROZEN_OUTPUT_DIR, FULL_OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

FAST_DEBUG = False
REBUILD_DATA = True
RUN_FORECAST_PRETRAIN = True
RUN_FROZEN_FINETUNE = True
RUN_FULL_FINETUNE = True

NUM_EPOCHS_FORECAST_PRETRAIN = 1 if FAST_DEBUG else 30
NUM_EPOCHS_FINETUNE = 1 if FAST_DEBUG else 20
BATCH_SIZE = 32 if FAST_DEBUG else 128
SMOKE_ENTITY_LIMIT = 64 if FAST_DEBUG else 512
SMOKE_BATCH_SIZE = 4 if FAST_DEBUG else 8

RAW_DATA_PATH = DATA_ROOT / "rosbank" / "train.csv.gz"


def run_logged(cmd: list[str], log_path: pathlib.Path) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", " ".join(map(str, cmd)))
    with log_path.open("w", encoding="utf-8") as f:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            f.write(line)
            f.flush()
        return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    print(f"Log saved: {log_path}")


def load_jsonl(path: pathlib.Path) -> list[dict]:
    if not path.exists():
        print(f"Metrics file is not available yet: {path}")
        return []
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def show_metrics_tail(path: pathlib.Path, title: str, tail: int = 8) -> pd.DataFrame:
    rows = load_jsonl(path)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    csv_path = path.with_suffix(".csv")
    df.to_csv(csv_path, index=False)
    print(f"{title}")
    print(f"JSONL: {path}")
    print(f"CSV  : {csv_path}")
    view = df.tail(tail)
    try:
        display(view)
    except NameError:
        print(view.to_string(index=False))
    return df


def find_raw_data(default_path: pathlib.Path) -> pathlib.Path:
    if default_path.exists():
        return default_path
    kaggle_input = pathlib.Path("/kaggle/input")
    candidates = sorted(kaggle_input.glob("**/train.csv.gz")) if kaggle_input.exists() else []
    if len(candidates) == 1:
        return candidates[0]
    if candidates:
        print("Multiple train.csv.gz candidates found:")
        for candidate in candidates:
            print(" -", candidate)
    raise FileNotFoundError(
        f"Raw data not found at {default_path}. Set RAW_DATA_PATH to the Kaggle file."
    )


print(f"Processed data : {PROCESSED_DIR}")
print(f"Outputs        : {OUTPUT_DIR}")
print(f"Config path    : {CONFIG_PATH}")
print(f"Fast debug     : {FAST_DEBUG}")

In [ ]:
# Cell 3 - Forecast Pipeline Config
from src.utils.config import load_experiment_config


def deep_update(base: dict, override: dict) -> dict:
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            deep_update(base[key], value)
        else:
            base[key] = value
    return base


config = load_experiment_config(base_path=str(REPO_DIR / "configs/base.yaml"))
deep_update(
    config,
    {
        "data": {
            "max_seq_len": 256,
            "min_seq_len": 5,
            "event_type_col": "MCC",
            "timestamp_col": "TRDATETIME",
            "target_col": "target_flag",
            "numerical_cols": ["amount"],
            "categorical_cols": ["channel_type", "trx_category"],
            "group_col": "cl_id",
            "truncation_pretrain": "sliding_window",
            "truncation_eval": "last_events",
            "amount_transform": "robust_scaler",
            "amount_cols": ["amount"],
            "robust_scale_cols": ["amount"],
            "time_transform": "none",
            "min_vocab_count": 5,
            "train_ratio": 0.70,
            "val_ratio": 0.15,
            "test_ratio": 0.15,
            "use_time_features": True,
            "time_feature_mode": "derived_numeric",
        },
        "model": {
            "max_seq_len": 256,
            "use_profile_token": True,
            "client_embedding_dim": 512,
        },
        "pooling": {
            "type": "multi",
            "components": ["profile", "gated_attention", "mean", "max", "last"],
        },
        "forecasting": {
            "enabled": True,
            "cut_min_ratio": 0.50,
            "cut_max_ratio": 0.80,
            "min_future_events": 2,
            "event_type_target_mode": "log_future_global_ratio",
            "count_num_buckets": 6,
            "gap_num_buckets": 6,
            "alpha_denoising": 0.2,
            "lambda_event_type_profile": 1.0,
            "lambda_count": 0.5,
            "lambda_amount": 0.5,
            "lambda_gap": 0.3,
            "lambda_cat_profile": 0.5,
            "finetune_aux_enabled": False,
            "finetune_aux_weight": 0.05,
        },
        "training": {
            "task": "binary",
            "num_classes": 2,
            "main_metrics": ["roc_auc", "pr_auc", "macro_f1"],
            "batch_size": BATCH_SIZE,
            "num_epochs_pretrain": NUM_EPOCHS_FORECAST_PRETRAIN,
            "num_epochs_finetune": NUM_EPOCHS_FINETUNE,
            "mixed_precision": USE_AMP,
            "early_stopping_patience": 5,
            "log_every_n_steps": 50,
        },
    },
)

with CONFIG_PATH.open("w") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=False)

print(f"Saved forecast config: {CONFIG_PATH}")
print(
    yaml.safe_dump(
        {
            "data": {
                "use_time_features": config["data"]["use_time_features"],
                "categorical_cols": config["data"]["categorical_cols"],
            },
            "model": {
                "use_profile_token": config["model"]["use_profile_token"],
                "client_embedding_dim": config["model"]["client_embedding_dim"],
            },
            "pooling": config["pooling"],
            "forecasting": config["forecasting"],
        },
        sort_keys=False,
    )
)

In [ ]:
# Cell 4 - Data Loading & Preprocessing Artifacts
from src.data.preprocessing import EventPreprocessor
from src.data.splits import make_entity_splits


def load_raw_rosbank(path: pathlib.Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    ts_col = config["data"]["timestamp_col"]
    df[ts_col] = pd.to_datetime(df[ts_col], format="%d%b%y:%H:%M:%S", errors="coerce")
    if df[ts_col].isna().any():
        df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce")
    if df[ts_col].isna().any():
        bad = int(df[ts_col].isna().sum())
        raise ValueError(f"Failed to parse {bad} timestamps in {ts_col}")
    return df


def write_raw_training_artifacts(
    raw_df: pd.DataFrame,
    splits: dict | None = None,
    prep: EventPreprocessor | None = None,
) -> tuple[pd.DataFrame, dict, EventPreprocessor]:
    data_cfg = config["data"]
    entity_col = data_cfg["group_col"]
    timestamp_col = data_cfg["timestamp_col"]
    target_col = data_cfg["target_col"]
    seed = int(config.get("seed", {}).get("global_seed", 42))

    df_sorted = raw_df.sort_values([entity_col, timestamp_col], kind="stable").reset_index(drop=True)
    helper = EventPreprocessor(config)
    df_sorted["time_delta"] = helper.compute_time_delta(df_sorted, entity_col, timestamp_col)

    if splits is None:
        splits = make_entity_splits(
            df_sorted,
            entity_col=entity_col,
            target_col=target_col,
            train_ratio=float(data_cfg.get("train_ratio", 0.70)),
            val_ratio=float(data_cfg.get("val_ratio", 0.15)),
            test_ratio=float(data_cfg.get("test_ratio", 0.15)),
            seed=seed,
            stratify=True,
        )
    if prep is None:
        prep = EventPreprocessor(config)
        prep.fit(df_sorted, splits["train"])

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    df_sorted.to_parquet(PROCESSED_DIR / "events.parquet", index=False)
    with (PROCESSED_DIR / "preprocessor.pkl").open("wb") as f:
        pickle.dump(prep, f, protocol=pickle.HIGHEST_PROTOCOL)
    with (PROCESSED_DIR / "splits.json").open("w") as f:
        json.dump(splits, f, indent=2)
    return df_sorted, splits, prep


artifacts_exist = all(
    (PROCESSED_DIR / name).exists()
    for name in ["events.parquet", "splits.json", "preprocessor.pkl"]
)

if artifacts_exist and not REBUILD_DATA:
    df_events = pd.read_parquet(PROCESSED_DIR / "events.parquet")
    with (PROCESSED_DIR / "splits.json").open() as f:
        splits = json.load(f)
    with (PROCESSED_DIR / "preprocessor.pkl").open("rb") as f:
        prep = pickle.load(f)
    print("Loaded existing prepared artifacts")
else:
    RAW_DATA_PATH = find_raw_data(RAW_DATA_PATH)
    raw_df = load_raw_rosbank(RAW_DATA_PATH)
    RAW_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
    raw_df.to_parquet(RAW_PARQUET_PATH, index=False)
    print(f"Raw data: {raw_df.shape} from {RAW_DATA_PATH}")

    try:
        run_logged(
            [
                sys.executable,
                str(REPO_DIR / "scripts/prepare_data.py"),
                "--config",
                str(CONFIG_PATH),
                "--input",
                str(RAW_PARQUET_PATH),
                "--output-dir",
                str(PROCESSED_DIR),
            ],
            LOG_DIR / "prepare_data.log",
        )
        with (PROCESSED_DIR / "splits.json").open() as f:
            splits = json.load(f)
        with (PROCESSED_DIR / "preprocessor.pkl").open("rb") as f:
            prep = pickle.load(f)
        df_events, splits, prep = write_raw_training_artifacts(raw_df, splits=splits, prep=prep)
        print("Prepared artifacts with prepare_data.py; restored raw events.parquet for Dataset")
    except subprocess.CalledProcessError as exc:
        print(f"prepare_data.py failed with exit code {exc.returncode}; using inline artifact builder")
        df_events, splits, prep = write_raw_training_artifacts(raw_df)

print(f"Prepared events: {df_events.shape}")
print(f"Splits: train={len(splits['train'])}, val={len(splits['val'])}, test={len(splits['test'])}")

In [ ]:
# Cell 5 - Prepared Artifact Inspection
df_events = pd.read_parquet(PROCESSED_DIR / "events.parquet")
with (PROCESSED_DIR / "splits.json").open() as f:
    splits = json.load(f)
with (PROCESSED_DIR / "preprocessor.pkl").open("rb") as f:
    prep = pickle.load(f)

entity_col = config["data"]["group_col"]
target_col = config["data"]["target_col"]
event_type_col = config["data"]["event_type_col"]
seq_lens = df_events.groupby(entity_col).size()
class_counts = df_events.groupby(entity_col)[target_col].first().value_counts().sort_index()

print(f"Rows          : {len(df_events):,}")
print(f"Entities      : {df_events[entity_col].nunique():,}")
print(f"Sequence len  : min={seq_lens.min()}, p50={seq_lens.median():.0f}, max={seq_lens.max()}")
print(f"Class counts  : {dict(class_counts)}")
print(f"Event vocab   : {len(prep.vocab[event_type_col])}")
print(f"Cat vocabs    : {[len(prep.vocab[c]) for c in prep.categorical_cols]}")
print(f"Num cols      : {prep.numerical_cols}")
print(f"Cat cols      : {prep.categorical_cols}")

In [ ]:
# Cell 6 - Forecast Hybrid Smoke Test
from torch.utils.data import DataLoader

from src.corruption.pipeline import CorruptionPipeline
from src.data.collate import collate_fn
from src.data.dataset import EventSequenceDataset
from src.data.forecasting import build_forecast_stats, get_num_feature_dim
from src.models.dme_encoder import DMEEncoder
from src.training.losses import compute_forecast_loss, compute_pretraining_loss


def to_device(value, target_device):
    if isinstance(value, torch.Tensor):
        return value.to(target_device)
    if isinstance(value, dict):
        return {k: to_device(v, target_device) for k, v in value.items()}
    if isinstance(value, list):
        return [to_device(v, target_device) for v in value]
    return value


def build_vocab_info(preprocessor, cfg: dict) -> dict:
    return {
        "event_type_vocab_size": len(preprocessor.vocab[preprocessor.event_type_col]),
        "cat_vocab_sizes": [len(preprocessor.vocab[col]) for col in preprocessor.categorical_cols],
        "num_num_features": get_num_feature_dim(preprocessor, cfg),
        "num_classes": int(cfg.get("training", {}).get("num_classes", 2)),
    }


smoke_ids = splits["train"][: min(len(splits["train"]), SMOKE_ENTITY_LIMIT)]
forecast_stats = build_forecast_stats(df_events, smoke_ids, prep, config)
smoke_ds = EventSequenceDataset(
    df_events,
    smoke_ids,
    prep,
    config,
    mode="forecast",
    forecast_stats=forecast_stats,
)
assert len(smoke_ds) > 0, "No valid forecast samples in smoke subset"
smoke_loader = DataLoader(
    smoke_ds,
    batch_size=SMOKE_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
)
clean_batch = to_device(next(iter(smoke_loader)), device)

vocab_info = build_vocab_info(prep, config)
model = DMEEncoder(config, vocab_info).to(device)
pipe = CorruptionPipeline(
    config=config["corruption"],
    transition_matrix=None,
    vocab_sizes={
        "event_type": vocab_info["event_type_vocab_size"],
        "cat_features": vocab_info["cat_vocab_sizes"],
    },
    time_transform=config["data"].get("time_transform", "log1p"),
)

model.train()
corrupted_batch, denoise_targets, masks = pipe(clean_batch)
masks["attention_mask"] = corrupted_batch["attention_mask"]
pretrain_outputs = model(corrupted_batch, mode="pretrain")
denoise_loss = compute_pretraining_loss(pretrain_outputs, denoise_targets, masks, config)

forecast_outputs = model(clean_batch, mode="forecast")
forecast_loss = compute_forecast_loss(forecast_outputs, clean_batch["forecast_targets"], config)
hybrid_loss = denoise_loss["total"] + config["forecasting"]["alpha_denoising"] * forecast_loss["total"]
hybrid_loss.backward()

print(f"Smoke batch shape       : event_type={tuple(clean_batch['event_type'].shape)}")
print(f"Denoising loss          : {denoise_loss['total'].item():.4f}")
print(f"Forecast loss           : {forecast_loss['total'].item():.4f}")
print(f"Hybrid loss             : {hybrid_loss.item():.4f}")
print(f"Forecast output keys    : {sorted(forecast_outputs.keys())}")

del model, pipe, clean_batch, corrupted_batch, pretrain_outputs, forecast_outputs
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Cell 7 - Hybrid Forecast Pretraining
FORECAST_SUMMARY_PATH = (
    FORECAST_OUTPUT_DIR
    / "metrics"
    / f"{CONFIG_PATH.stem}_forecast_pretrain_summary.json"
)

if RUN_FORECAST_PRETRAIN:
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts/forecast_pretrain.py"),
        "--config",
        str(CONFIG_PATH),
        "--data-dir",
        str(PROCESSED_DIR),
        "--output-dir",
        str(FORECAST_OUTPUT_DIR),
    ]
    run_logged(cmd, LOG_DIR / "forecast_pretrain.log")
else:
    print("RUN_FORECAST_PRETRAIN=False; expecting an existing summary")

assert FORECAST_SUMMARY_PATH.exists(), f"Missing forecast summary: {FORECAST_SUMMARY_PATH}"
print(f"Forecast pretrain summary: {FORECAST_SUMMARY_PATH}")
FORECAST_METRICS_JSONL = (
    FORECAST_OUTPUT_DIR / "logs" / f"{CONFIG_PATH.stem}_forecast_pretrain" / "metrics.jsonl"
)
forecast_metrics_df = show_metrics_tail(FORECAST_METRICS_JSONL, "Forecast pretrain metrics")

In [ ]:
# Cell 8 - Select Forecast Checkpoint
forecast_summary = json.loads(FORECAST_SUMMARY_PATH.read_text())
FORECAST_CKPT_PATH = pathlib.Path(forecast_summary["best_checkpoint"])
FORECAST_STATS_PATH = pathlib.Path(forecast_summary["forecast_stats"])

assert FORECAST_CKPT_PATH.exists(), f"Missing checkpoint: {FORECAST_CKPT_PATH}"
assert FORECAST_STATS_PATH.exists(), f"Missing forecast stats: {FORECAST_STATS_PATH}"

print(f"Best forecast checkpoint: {FORECAST_CKPT_PATH}")
print(f"Forecast stats          : {FORECAST_STATS_PATH}")
print("Vocab info:", forecast_summary.get("vocab_info"))

In [ ]:
# Cell 9 - Fine-tuning Frozen Encoder
FROZEN_METRICS_PATH = (
    FROZEN_OUTPUT_DIR / "metrics" / f"{CONFIG_PATH.stem}_finetune_metrics.json"
)

if RUN_FROZEN_FINETUNE:
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts/finetune.py"),
        "--config",
        str(CONFIG_PATH),
        "--pretrained-checkpoint",
        str(FORECAST_CKPT_PATH),
        "--data-dir",
        str(PROCESSED_DIR),
        "--output-dir",
        str(FROZEN_OUTPUT_DIR),
        "--frozen-encoder",
    ]
    run_logged(cmd, LOG_DIR / "finetune_frozen.log")
else:
    print("RUN_FROZEN_FINETUNE=False; skipping frozen encoder fine-tune")

assert FROZEN_METRICS_PATH.exists(), f"Missing frozen metrics: {FROZEN_METRICS_PATH}"
print(f"Frozen encoder metrics: {FROZEN_METRICS_PATH}")
FROZEN_TRAIN_METRICS_JSONL = (
    FROZEN_OUTPUT_DIR / "logs" / f"{CONFIG_PATH.stem}_finetune" / "metrics.jsonl"
)
frozen_train_metrics_df = show_metrics_tail(
    FROZEN_TRAIN_METRICS_JSONL,
    "Frozen encoder fine-tune train/val metrics",
)
print("Frozen encoder test metrics:")
try:
    display(pd.DataFrame([json.loads(FROZEN_METRICS_PATH.read_text()).get("test_metrics", {})]))
except NameError:
    print(json.loads(FROZEN_METRICS_PATH.read_text()).get("test_metrics", {}))

In [ ]:
# Cell 10 - Full Fine-tuning
FULL_METRICS_PATH = FULL_OUTPUT_DIR / "metrics" / f"{CONFIG_PATH.stem}_finetune_metrics.json"

if RUN_FULL_FINETUNE:
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts/finetune.py"),
        "--config",
        str(CONFIG_PATH),
        "--pretrained-checkpoint",
        str(FORECAST_CKPT_PATH),
        "--data-dir",
        str(PROCESSED_DIR),
        "--output-dir",
        str(FULL_OUTPUT_DIR),
    ]
    run_logged(cmd, LOG_DIR / "finetune_full.log")
else:
    print("RUN_FULL_FINETUNE=False; skipping full fine-tune")

if RUN_FULL_FINETUNE:
    assert FULL_METRICS_PATH.exists(), f"Missing full fine-tune metrics: {FULL_METRICS_PATH}"
    print(f"Full fine-tune metrics: {FULL_METRICS_PATH}")
    FULL_TRAIN_METRICS_JSONL = (
        FULL_OUTPUT_DIR / "logs" / f"{CONFIG_PATH.stem}_finetune" / "metrics.jsonl"
    )
    full_train_metrics_df = show_metrics_tail(
        FULL_TRAIN_METRICS_JSONL,
        "Full fine-tune train/val metrics",
    )
    print("Full fine-tune test metrics:")
    try:
        display(pd.DataFrame([json.loads(FULL_METRICS_PATH.read_text()).get("test_metrics", {})]))
    except NameError:
        print(json.loads(FULL_METRICS_PATH.read_text()).get("test_metrics", {}))

In [ ]:
# Cell 11 - Results Table
def load_metric_row(path: pathlib.Path, experiment: str) -> dict | None:
    if not path.exists():
        return None
    payload = json.loads(path.read_text())
    row = {
        "experiment": experiment,
        "best_checkpoint": payload.get("best_checkpoint"),
    }
    row.update(payload.get("test_metrics", {}))
    return row


rows = []
for item in [
    load_metric_row(FROZEN_METRICS_PATH, "forecast_pretrain_frozen_encoder"),
    load_metric_row(FULL_METRICS_PATH, "forecast_pretrain_full_finetune"),
]:
    if item is not None:
        rows.append(item)

results_df = pd.DataFrame(rows)
RESULTS_CSV_PATH = OUTPUT_DIR / "results.csv"
RESULTS_JSON_PATH = OUTPUT_DIR / "results.json"
results_df.to_csv(RESULTS_CSV_PATH, index=False)
RESULTS_JSON_PATH.write_text(json.dumps(rows, indent=2, ensure_ascii=False))

metric_cols = [c for c in results_df.columns if c not in ("experiment", "best_checkpoint")]
print("Results saved:")
print(" -", RESULTS_CSV_PATH)
print(" -", RESULTS_JSON_PATH)

if len(results_df) >= 2 and "roc_auc" in results_df.columns:
    frozen = results_df[results_df["experiment"] == "forecast_pretrain_frozen_encoder"]
    full = results_df[results_df["experiment"] == "forecast_pretrain_full_finetune"]
    if len(frozen) == 1 and len(full) == 1:
        delta = full.iloc[0][metric_cols].astype(float) - frozen.iloc[0][metric_cols].astype(float)
        print("\nFull fine-tune minus frozen encoder:")
        print(delta.to_string(float_format=lambda x: f"{x:+.4f}"))

try:
    display(results_df)
except NameError:
    print(results_df.to_string(index=False))

In [ ]:
# Cell 12 - Artifact Summary
artifacts = {
    "config": CONFIG_PATH,
    "processed_data": PROCESSED_DIR,
    "subprocess_logs": LOG_DIR,
    "forecast_summary": FORECAST_SUMMARY_PATH,
    "forecast_checkpoint": FORECAST_CKPT_PATH,
    "forecast_stats": FORECAST_STATS_PATH,
    "forecast_metrics_jsonl": FORECAST_METRICS_JSONL,
    "frozen_metrics": FROZEN_METRICS_PATH,
    "frozen_train_metrics_jsonl": FROZEN_TRAIN_METRICS_JSONL,
    "full_metrics": FULL_METRICS_PATH if RUN_FULL_FINETUNE else None,
    "full_train_metrics_jsonl": FULL_TRAIN_METRICS_JSONL if RUN_FULL_FINETUNE else None,
    "results_csv": RESULTS_CSV_PATH,
    "results_json": RESULTS_JSON_PATH,
}

print("Pipeline artifacts")
for name, path in artifacts.items():
    if path is None:
        print(f"{name:20s}: skipped")
    else:
        print(f"{name:20s}: {path}")

In [ ]:
# Cell 13 - Low-label Grid Setup
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader

from src.data.collate import collate_fn
from src.data.dataset import EventSequenceDataset
from src.data.forecasting import get_num_feature_dim
from src.models.dme_encoder import DMEEncoder
from src.training.finetune import evaluate_finetune, finetune
from src.utils.logging import MetricsLogger
from src.utils.seed import set_seed

try:
    vocab_info
except NameError:
    vocab_info = {
        "event_type_vocab_size": len(prep.vocab[prep.event_type_col]),
        "cat_vocab_sizes": [len(prep.vocab[col]) for col in prep.categorical_cols],
        "num_num_features": get_num_feature_dim(prep, config),
        "num_classes": int(config.get("training", {}).get("num_classes", 2)),
    }

LOW_LABEL_SEEDS = [42, 43, 44, 45, 46]
LOW_LABEL_FRACTIONS = [0.05, 0.25, 0.50, 0.75, 1.00]
LOW_LABEL_PIPELINES = ["frozen_head", "full_finetune", "embedding_catboost"]
RUN_LOW_LABEL_GRID = True
RESUME_LOW_LABEL_GRID = True
LOW_LABEL_OUTPUT_DIR = OUTPUT_DIR / "low_label_grid"
LOW_LABEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOW_LABEL_FINETUNE_EPOCHS = config["training"].get("num_epochs_finetune", NUM_EPOCHS_FINETUNE)
LOW_LABEL_BATCH_SIZE = config["training"].get("batch_size", BATCH_SIZE)
LOW_LABEL_CATBOOST_ITERATIONS = 2500
LOW_LABEL_CATBOOST_VERBOSE = 250


def labels_by_entity_from_events(df_events: pd.DataFrame) -> dict[str, int]:
    entity_col = config["data"]["group_col"]
    target_col = config["data"]["target_col"]
    return {
        str(k): int(v)
        for k, v in df_events.groupby(entity_col)[target_col].first().to_dict().items()
    }


def sample_labeled_entities(
    train_ids: list,
    labels_by_entity: dict[str, int],
    fraction: float,
    seed: int,
) -> list:
    train_ids = list(train_ids)
    if fraction >= 1.0:
        return train_ids
    rng = np.random.default_rng(seed)
    by_label: dict[int, list] = {}
    for entity_id in train_ids:
        by_label.setdefault(int(labels_by_entity[str(entity_id)]), []).append(entity_id)
    sampled: list = []
    for label, ids in sorted(by_label.items()):
        ids_arr = np.asarray(ids, dtype=object)
        n = max(1, int(round(len(ids_arr) * fraction)))
        n = min(n, len(ids_arr))
        sampled.extend(rng.choice(ids_arr, size=n, replace=False).tolist())
    rng.shuffle(sampled)
    return sampled


def metric_row_from_json(path: pathlib.Path) -> dict | None:
    if not path.exists():
        return None
    return json.loads(path.read_text())


def build_eval_loader(entity_ids: list) -> DataLoader:
    entity_ids = normalize_entity_ids(entity_ids)
    dataset = EventSequenceDataset(df_events, entity_ids, prep, config, mode="eval")
    if len(dataset) == 0:
        raise ValueError(
            "EventSequenceDataset is empty. Check cl_id types in splits vs df_events. "
            f"Example ids: {entity_ids[:5]}"
        )
    return DataLoader(
        dataset,
        batch_size=LOW_LABEL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
    )


labels_by_entity = labels_by_entity_from_events(df_events)
entity_col = config["data"]["group_col"]
entity_id_lookup = {str(x): x for x in df_events[entity_col].drop_duplicates().tolist()}
splits_str = {k: [str(x) for x in v] for k, v in splits.items()}


def normalize_entity_ids(entity_ids: list) -> list:
    return [entity_id_lookup.get(str(x), x) for x in entity_ids]


ll_val_loader = build_eval_loader(splits["val"])
ll_test_loader = build_eval_loader(splits["test"])

print("Low-label grid")
print("Pipelines :", LOW_LABEL_PIPELINES)
print("Fractions :", LOW_LABEL_FRACTIONS)
print("Seeds     :", LOW_LABEL_SEEDS)
print("Output    :", LOW_LABEL_OUTPUT_DIR)


In [ ]:
# Cell 14 - Low-label Neural Grid: Frozen Head + Full Fine-tune
LOW_LABEL_NEURAL_ROWS: list[dict] = []


def run_low_label_neural_once(pipeline: str, fraction: float, seed: int) -> dict:
    frozen_encoder = pipeline == "frozen_head"
    run_id = f"{pipeline}_frac{fraction:.2f}_seed{seed}"
    run_dir = LOW_LABEL_OUTPUT_DIR / pipeline / f"frac_{fraction:.2f}" / f"seed_{seed}"
    metrics_path = run_dir / "test_metrics.json"
    if RESUME_LOW_LABEL_GRID and metrics_path.exists():
        row = json.loads(metrics_path.read_text())
        print(f"[skip] {run_id} -> roc_auc={row.get('roc_auc', float('nan')):.4f}")
        return row

    print(f"\n=== {run_id} ===")
    set_seed(seed)
    train_ids = sample_labeled_entities(splits["train"], labels_by_entity, fraction, seed)
    train_loader = DataLoader(
        EventSequenceDataset(df_events, train_ids, prep, config, mode="finetune"),
        batch_size=LOW_LABEL_BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
    )

    run_cfg = json.loads(json.dumps(config))
    run_cfg["seed"] = {**run_cfg.get("seed", {}), "global_seed": seed}
    run_cfg["training"] = {
        **run_cfg.get("training", {}),
        "num_epochs_finetune": LOW_LABEL_FINETUNE_EPOCHS,
        "batch_size": LOW_LABEL_BATCH_SIZE,
    }

    model = DMEEncoder(run_cfg, vocab_info).to(device)
    logger = MetricsLogger(str(run_dir / "logs"), run_id)
    logger.log_config(run_cfg)
    best_ckpt = finetune(
        model=model,
        train_loader=train_loader,
        val_loader=ll_val_loader,
        config=run_cfg,
        output_dir=str(run_dir / "checkpoints"),
        device=device,
        logger=logger,
        pretrained_checkpoint=str(FORECAST_CKPT_PATH),
        frozen_encoder=frozen_encoder,
        label_fraction=1.0,
        vocab_info=vocab_info,
    )

    ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    metrics = evaluate_finetune(model, ll_test_loader, num_classes=vocab_info["num_classes"], device=device)
    row = {
        "pipeline": pipeline,
        "fraction": fraction,
        "seed": seed,
        "train_entities": len(train_ids),
        "best_checkpoint": str(best_ckpt),
        **metrics,
    }
    run_dir.mkdir(parents=True, exist_ok=True)
    metrics_path.write_text(json.dumps(row, indent=2, ensure_ascii=False))
    print(row)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return row


if RUN_LOW_LABEL_GRID:
    for pipeline in ["frozen_head", "full_finetune"]:
        for fraction in LOW_LABEL_FRACTIONS:
            for seed in LOW_LABEL_SEEDS:
                LOW_LABEL_NEURAL_ROWS.append(run_low_label_neural_once(pipeline, fraction, seed))
else:
    print("RUN_LOW_LABEL_GRID=False; loading existing neural low-label rows")
    for pipeline in ["frozen_head", "full_finetune"]:
        for fraction in LOW_LABEL_FRACTIONS:
            for seed in LOW_LABEL_SEEDS:
                path = LOW_LABEL_OUTPUT_DIR / pipeline / f"frac_{fraction:.2f}" / f"seed_{seed}" / "test_metrics.json"
                row = metric_row_from_json(path)
                if row is not None:
                    LOW_LABEL_NEURAL_ROWS.append(row)

low_label_neural_df = pd.DataFrame(LOW_LABEL_NEURAL_ROWS)
low_label_neural_path = LOW_LABEL_OUTPUT_DIR / "low_label_neural_runs.csv"
low_label_neural_df.to_csv(low_label_neural_path, index=False)
print(f"Neural low-label runs saved: {low_label_neural_path}")
print(low_label_neural_df.to_string(index=False))


In [ ]:
# Cell 15 - Low-label Embeddings + CatBoost Grid
try:
    from catboost import CatBoostClassifier
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "catboost", "--quiet"], check=True)
    from catboost import CatBoostClassifier

LOW_LABEL_CATBOOST_ROWS: list[dict] = []
LOW_LABEL_CATBOOST_PREDICTIONS: list[pd.DataFrame] = []


def compute_binary_metrics(y_true, pos_proba) -> dict:
    y_true = np.asarray(y_true).astype(int)
    pos_proba = np.asarray(pos_proba).astype(float)
    y_pred = (pos_proba >= 0.5).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, pos_proba)),
        "pr_auc": float(average_precision_score(y_true, pos_proba)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


def extract_dme_embeddings(model: DMEEncoder, loader: DataLoader) -> pd.DataFrame:
    rows = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            entity_ids = [str(x) for x in batch["entity_id"]]
            labels = batch["label"].cpu().numpy().astype(int)
            batch_device = {
                k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()
            }
            reps = model(batch_device, mode="encode")["representation"].detach().cpu().numpy()
            for entity_id, label, rep in zip(entity_ids, labels, reps):
                row = {"entity_id": entity_id, "target": int(label)}
                row.update({f"emb_{idx:04d}": float(value) for idx, value in enumerate(rep)})
                rows.append(row)
    if not rows:
        raise ValueError(
            "No embeddings were extracted. The loader is empty; rerun Cell 13 so "
            "build_eval_loader normalizes split ids to the original cl_id dtype."
        )
    return pd.DataFrame(rows).set_index("entity_id")


embedding_dir = LOW_LABEL_OUTPUT_DIR / "embeddings"
embedding_dir.mkdir(parents=True, exist_ok=True)
train_emb_path = embedding_dir / "train_embeddings.parquet"
val_emb_path = embedding_dir / "val_embeddings.parquet"
test_emb_path = embedding_dir / "test_embeddings.parquet"

use_cached_embeddings = all(path.exists() for path in [train_emb_path, val_emb_path, test_emb_path])
if use_cached_embeddings:
    train_emb = pd.read_parquet(train_emb_path)
    val_emb = pd.read_parquet(val_emb_path)
    test_emb = pd.read_parquet(test_emb_path)
    if "entity_id" in train_emb.columns:
        train_emb = train_emb.set_index("entity_id")
    if "entity_id" in val_emb.columns:
        val_emb = val_emb.set_index("entity_id")
    if "entity_id" in test_emb.columns:
        test_emb = test_emb.set_index("entity_id")
    train_emb.index = train_emb.index.astype(str)
    val_emb.index = val_emb.index.astype(str)
    test_emb.index = test_emb.index.astype(str)
    cached_feature_cols = [c for c in train_emb.columns if c.startswith("emb_")]
    use_cached_embeddings = bool(cached_feature_cols) and len(train_emb) > 0 and len(val_emb) > 0 and len(test_emb) > 0
    if use_cached_embeddings:
        print("Loaded cached DME embeddings")
    else:
        print("Cached DME embeddings are empty or invalid; regenerating")
if not use_cached_embeddings:
    print("Extracting frozen DME embeddings from forecast checkpoint")
    embed_model = DMEEncoder(config, vocab_info).to(device)
    embed_ckpt = torch.load(FORECAST_CKPT_PATH, map_location=device, weights_only=False)
    embed_model.load_state_dict(embed_ckpt["model_state_dict"], strict=False)
    embed_train_loader = build_eval_loader(splits["train"])
    embed_val_loader = build_eval_loader(splits["val"])
    embed_test_loader = build_eval_loader(splits["test"])
    train_emb = extract_dme_embeddings(embed_model, embed_train_loader)
    val_emb = extract_dme_embeddings(embed_model, embed_val_loader)
    test_emb = extract_dme_embeddings(embed_model, embed_test_loader)
    train_emb.to_parquet(train_emb_path)
    val_emb.to_parquet(val_emb_path)
    test_emb.to_parquet(test_emb_path)
    del embed_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

feature_cols = [c for c in train_emb.columns if c.startswith("emb_")]
X_val = val_emb[feature_cols]
y_val = val_emb["target"].astype(int)
X_test = test_emb[feature_cols]
y_test = test_emb["target"].astype(int)


def run_catboost_once(fraction: float, seed: int) -> dict:
    pipeline = "embedding_catboost"
    run_id = f"{pipeline}_frac{fraction:.2f}_seed{seed}"
    run_dir = LOW_LABEL_OUTPUT_DIR / pipeline / f"frac_{fraction:.2f}" / f"seed_{seed}"
    metrics_path = run_dir / "test_metrics.json"
    pred_path = run_dir / "test_predictions.csv"
    if RESUME_LOW_LABEL_GRID and metrics_path.exists():
        row = json.loads(metrics_path.read_text())
        print(f"[skip] {run_id} -> roc_auc={row.get('roc_auc', float('nan')):.4f}")
        return row

    train_ids = [str(x) for x in sample_labeled_entities(splits["train"], labels_by_entity, fraction, seed)]
    X_train = train_emb.loc[train_ids, feature_cols]
    y_train = train_emb.loc[train_ids, "target"].astype(int)
    print(f"\n=== {run_id} | train_entities={len(train_ids)} ===")
    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=LOW_LABEL_CATBOOST_ITERATIONS,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5.0,
        random_seed=seed,
        auto_class_weights="Balanced",
        od_type="Iter",
        od_wait=150,
        allow_writing_files=False,
        verbose=LOW_LABEL_CATBOOST_VERBOSE,
    )
    model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
    test_proba = model.predict_proba(X_test)[:, 1]
    metrics = compute_binary_metrics(y_test.values, test_proba)
    row = {
        "pipeline": pipeline,
        "fraction": fraction,
        "seed": seed,
        "train_entities": len(train_ids),
        **metrics,
    }
    run_dir.mkdir(parents=True, exist_ok=True)
    metrics_path.write_text(json.dumps(row, indent=2, ensure_ascii=False))
    pd.DataFrame(
        {
            config["data"]["group_col"]: X_test.index,
            "target": y_test.values,
            "proba": test_proba,
            "fraction": fraction,
            "seed": seed,
        }
    ).to_csv(pred_path, index=False)
    print(row)
    return row


if RUN_LOW_LABEL_GRID:
    for fraction in LOW_LABEL_FRACTIONS:
        for seed in LOW_LABEL_SEEDS:
            LOW_LABEL_CATBOOST_ROWS.append(run_catboost_once(fraction, seed))
else:
    print("RUN_LOW_LABEL_GRID=False; loading existing CatBoost low-label rows")
    for fraction in LOW_LABEL_FRACTIONS:
        for seed in LOW_LABEL_SEEDS:
            path = LOW_LABEL_OUTPUT_DIR / "embedding_catboost" / f"frac_{fraction:.2f}" / f"seed_{seed}" / "test_metrics.json"
            row = metric_row_from_json(path)
            if row is not None:
                LOW_LABEL_CATBOOST_ROWS.append(row)

low_label_catboost_df = pd.DataFrame(LOW_LABEL_CATBOOST_ROWS)
low_label_catboost_path = LOW_LABEL_OUTPUT_DIR / "low_label_catboost_runs.csv"
low_label_catboost_df.to_csv(low_label_catboost_path, index=False)
print(f"CatBoost low-label runs saved: {low_label_catboost_path}")
print(low_label_catboost_df.to_string(index=False))


In [ ]:
# Cell 16 - Low-label Aggregate Results
LOW_LABEL_METRICS = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "roc_auc",
    "pr_auc",
    "balanced_accuracy",
]

all_low_label_runs = pd.concat(
    [low_label_neural_df, low_label_catboost_df],
    ignore_index=True,
    sort=False,
)
all_runs_path = LOW_LABEL_OUTPUT_DIR / "low_label_all_runs.csv"
all_low_label_runs.to_csv(all_runs_path, index=False)

agg_rows = []
for (pipeline, fraction), group in all_low_label_runs.groupby(["pipeline", "fraction"], sort=True):
    row = {"pipeline": pipeline, "fraction": float(fraction), "n_seeds": int(group["seed"].nunique())}
    for metric in LOW_LABEL_METRICS:
        values = pd.to_numeric(group[metric], errors="coerce").dropna()
        if len(values):
            row[f"{metric}_mean"] = float(values.mean())
            row[f"{metric}_std"] = float(values.std(ddof=0))
    agg_rows.append(row)

low_label_agg_df = pd.DataFrame(agg_rows).sort_values(["fraction", "pipeline"]).reset_index(drop=True)
agg_path = LOW_LABEL_OUTPUT_DIR / "low_label_aggregate.csv"
agg_json_path = LOW_LABEL_OUTPUT_DIR / "low_label_aggregate.json"
low_label_agg_df.to_csv(agg_path, index=False)
agg_json_path.write_text(json.dumps(agg_rows, indent=2, ensure_ascii=False))

summary_rows = []
for row in agg_rows:
    summary_rows.append(
        {
            "pipeline": row["pipeline"],
            "fraction": row["fraction"],
            "n_seeds": row["n_seeds"],
            "roc_auc": f"{row.get('roc_auc_mean', float('nan')):.4f} +/- {row.get('roc_auc_std', 0.0):.4f}",
            "pr_auc": f"{row.get('pr_auc_mean', float('nan')):.4f} +/- {row.get('pr_auc_std', 0.0):.4f}",
            "macro_f1": f"{row.get('macro_f1_mean', float('nan')):.4f} +/- {row.get('macro_f1_std', 0.0):.4f}",
            "balanced_accuracy": f"{row.get('balanced_accuracy_mean', float('nan')):.4f} +/- {row.get('balanced_accuracy_std', 0.0):.4f}",
        }
    )
summary_df = pd.DataFrame(summary_rows).sort_values(["fraction", "pipeline"]).reset_index(drop=True)
summary_path = LOW_LABEL_OUTPUT_DIR / "low_label_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Low-label run results saved:")
print(" -", all_runs_path)
print(" -", agg_path)
print(" -", agg_json_path)
print(" -", summary_path)
print("\nLow-label summary:")
print(summary_df.to_string(index=False))

best_by_fraction = (
    low_label_agg_df.sort_values(["fraction", "roc_auc_mean"], ascending=[True, False])
    .groupby("fraction", as_index=False)
    .head(1)[["fraction", "pipeline", "roc_auc_mean", "roc_auc_std", "pr_auc_mean", "pr_auc_std"]]
)
print("\nBest pipeline by fraction, ranked by ROC-AUC mean:")
print(best_by_fraction.to_string(index=False))
